# Credit Card Customer Segmentation Using Clustering

## 1. Import Required Libraries

This notebook applies unsupervised learning to identify customer segments from credit card usage behavior.

Pandas and NumPy are used for data handling, Matplotlib and Seaborn for visualization, and Scikit-learn for preprocessing, clustering, evaluation and dimensionality reduction. SciPy is used to construct hierarchical dendrograms.

A fixed random seed is used wherever sampling or randomized dimensionality reduction is involved so that the experiments can be reproduced.

In [7]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.pipeline import Pipeline

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from scipy.cluster.hierarchy import linkage, dendrogram

RANDOM_STATE = 42

sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load the Credit Card Dataset

The dataset contains customer-level information such as balances, purchases, payments, credit limits, cash advances and purchase frequency.

The first step is to load the dataset and inspect its dimensions, column names, data types and sample records. This verifies that the file has been loaded correctly before preprocessing begins.

In [9]:
# Update this path if your CSV has a different filename or location
df = pd.read_csv("C:\\Users\\KARTHIKKHA SHREE\\OneDrive\\Desktop\\ML_CaseStudy\\data\\CC_GENERAL.csv")

print("Dataset shape:", df.shape)

display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

Dataset shape: (8950, 18)


,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12



Column names:
['CUST_ID', 'BALANCE', 'BALANCE_FREQUENCY', 'PURCHASES', 'ONEOFF_PURCHASES', 'INSTALLMENTS_PURCHASES', 'CASH_ADVANCE', 'PURCHASES_FREQUENCY', 'ONEOFF_PURCHASES_FREQUENCY', 'PURCHASES_INSTALLMENTS_FREQUENCY', 'CASH_ADVANCE_FREQUENCY', 'CASH_ADVANCE_TRX', 'PURCHASES_TRX', 'CREDIT_LIMIT', 'PAYMENTS', 'MINIMUM_PAYMENTS', 'PRC_FULL_PAYMENT', 'TENURE']

Data types:
CUST_ID                                 str
BALANCE                             float64
BALANCE_FREQUENCY                   float64
PURCHASES                           float64
ONEOFF_PURCHASES                    float64
INSTALLMENTS_PURCHASES              float64
CASH_ADVANCE                        float64
PURCHASES_FREQUENCY                 float64
ONEOFF_PURCHASES_FREQUENCY          float64
PURCHASES_INSTALLMENTS_FREQUENCY    float64
CASH_ADVANCE_FREQUENCY              float64
CASH_ADVANCE_TRX                      int64
PURCHASES_TRX                         int64
CREDIT_LIMIT                        float64
PAYMENT

### Dataset Loading Observation

The dataset shape indicates the number of customer records and available attributes. The column names and data types help identify numerical financial attributes and any identifier columns.

The dataset is inspected before preprocessing so that missing values, identifier columns and unsuitable features can be handled appropriately.

## 3. Data Quality Audit

Missing values can affect distance calculations and prevent clustering algorithms from operating correctly. Duplicate records may also influence the identified customer groups.

This step examines missing values, duplicate rows, descriptive statistics and the presence of a customer identifier. The identifier `CUST_ID`, if present, is retained separately for reference but excluded from the clustering features because its numerical or categorical representation does not describe financial behavior.

In [10]:
# Preserve a working copy
df_cluster = df.copy()

# Remove accidental whitespace from column names
df_cluster.columns = df_cluster.columns.str.strip()

print("Missing values per column:")
display(
    df_cluster.isnull().sum()
    .sort_values(ascending=False)
    .to_frame("Missing Count")
)

print("\nDuplicate rows:", df_cluster.duplicated().sum())

print("\nDescriptive statistics:")
display(df_cluster.describe().T)

# Keep customer identifiers separately
if "CUST_ID" in df_cluster.columns:
    customer_ids = df_cluster["CUST_ID"].copy()
    df_cluster = df_cluster.drop(columns=["CUST_ID"])
else:
    customer_ids = pd.Series(
        df_cluster.index,
        index=df_cluster.index,
        name="Customer_Index"
    )

print("\nShape after excluding customer ID:", df_cluster.shape)

Missing values per column:


,Missing Count
MINIMUM_PAYMENTS,313
CREDIT_LIMIT,1
BALANCE,0
CUST_ID,0
BALANCE_FREQUENCY,0
PURCHASES,0
CASH_ADVANCE,0
PURCHASES_FREQUENCY,0
ONEOFF_PURCHASES,0
INSTALLMENTS_PURCHASES,0



Duplicate rows: 0

Descriptive statistics:


,count,mean,std,min,25%,50%,75%,max
BALANCE,8950.0,1564.474828,2081.531879,0.000000,128.281915,873.385231,2054.140036,19043.13856
BALANCE_FREQUENCY,8950.0,0.877271,0.236904,0.000000,0.888889,1.000000,1.000000,1.00000
PURCHASES,8950.0,1003.204834,2136.634782,0.000000,39.635000,361.280000,1110.130000,49039.57000
ONEOFF_PURCHASES,8950.0,592.437371,1659.887917,0.000000,0.000000,38.000000,577.405000,40761.25000
INSTALLMENTS_PURCHASES,8950.0,411.067645,904.338115,0.000000,0.000000,89.000000,468.637500,22500.00000
CASH_ADVANCE,8950.0,978.871112,2097.163877,0.000000,0.000000,0.000000,1113.821139,47137.21176
PURCHASES_FREQUENCY,8950.0,0.490351,0.401371,0.000000,0.083333,0.500000,0.916667,1.00000
ONEOFF_PURCHASES_FREQUENCY,8950.0,0.202458,0.298336,0.000000,0.000000,0.083333,0.300000,1.00000
PURCHASES_INSTALLMENTS_FREQUENCY,8950.0,0.364437,0.397448,0.000000,0.000000,0.166667,0.750000,1.00000
CASH_ADVANCE_FREQUENCY,8950.0,0.135144,0.200121,0.000000,0.000000,0.000000,0.222222,1.50000



Shape after excluding customer ID: (8950, 17)


### Data Audit Observation

The audit identifies attributes with missing values and reveals the ranges and distributions of financial variables. Missing values must be handled before calculating distances between customers.

Customer IDs are not valid behavioral features, so they are excluded. Duplicate records are checked and will be removed if confirmed to be accidental duplicates. Records are not removed solely because their financial values are unusual; extreme values may represent genuine customer behavior.